## Fact Table Parameters

In [0]:
#Catalog
catalog = "dataengineerflightproject"

##Source##
source_object = "silver_bookings"
source_schema = "silver"

#CDC Column
cdc_col = "modified_date"

#Backdated Refresh
backdated_refresh = ""

#Source Fact Table
fact_table = f"{catalog}.{source_schema}.{source_object}"

###Target Schema#
target_object = "fact_bookings"
target_schema = "gold"

#Fact key column list
fact_key_columns = ["dim_passengers_key", "dim_flights_key", "dim_airports_key", "booking_date"]

## Dimenstion Dictionary


In [0]:
##Setup 3 Dimension Table
dimensions = [
    {
        "table": f"{catalog}.{target_schema}.dim_passengers",
        "alias": "dim_passengers",
        "join_keys": ["passenger_id", "passenger_id"] #fact_col and dim_col when join on...
    },
    
    {
        "table": f"{catalog}.{target_schema}.dim_flights",
        "alias": "dim_flights",
        "surrogate_key": "dim_flight_key",
        "join_keys": ["flight_id", "flight_id"] #fact_col and dim_col when join on...
    },

    {
        "table": f"{catalog}.{target_schema}.dim_airports",
        "alias": "dim_airports",
        "join_keys": ["airport_id", "airport_id"] #fact_col and dim_col when join on...
    }
]

## Column that want to keep from fact table
fact_column = ["amount", "booking_date", "modified_date"]

## Last Load Date

In [0]:
##Check the last load date with conditional statment
if len(backdated_refresh) == 0: ## No backdated refresh

    if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"): #But already have a table at gold
       last_load_date = spark.sql(f"SELECT max({cdc_col}) FROM {catalog}.{target_schema}.{target_object}").collect()[0][0]
       #Pull a latest timestamp from the target table

    else: ##No table at gold
       last_load_date = "1900-01-01 00:00:00"

else: ##Add variable last_load_date as the same as the original
    last_load_date = backdated_refresh

#Test
last_load_date

## Dynamic Fact Query

In [0]:
def generate_fact_query_incremental(fact_table, dimensions, fact_columns, cdc_column, processing_date):
    fact_alias = "f"

    #Base column to select
    select_cols = [f"{fact_alias}.{col}" for col in fact_columns]

    #Build joins dynamically
    join_clauses = []
    for dim in dimensions:
        table_full = dim["table"] #table full name
        dim_alias = dim["alias"] #make the name table as
        table_name = table_full.split(".")[-1] #get the table name
        surrogate_key = f"{dim_alias}.{table_name}_key" #what is surrogate key

        join_keys = dim["join_keys"] #get join condition
        #join_key_pairs = [f"{fact_alias}.{join_key} = {dim_alias}.{join_key}" for join_key in join_keys]
        select_cols.append(surrogate_key)
       
        #Build ON clause

        #on_conditions = [f"{fact_alias}.{fk} = {dim_alias}.{dk}" for fk, dk in dim["join_keys"]]

        #Better Version with single/double join key case
        if join_keys and isinstance(join_keys[0], tuple):
            on_conditions = [f"{fact_alias}.{fk} = {dim_alias}.{dk}" for fk, dk in join_keys]
        else:
            on_conditions = [f"{fact_alias}.{jk} = {dim_alias}.{jk}" for jk in join_keys]

        join_clause = f"JOIN {table_full} {dim_alias} ON {' AND '.join(on_conditions)}"
        join_clauses.append(join_clause)

    #Final SELECT and Join clause
    select_clause = ",\n    ".join(select_cols)
    join_clause = "\n".join(join_clauses)

    #Where clause
    where_clause = f"{fact_alias}.{cdc_column} >= DATE('{processing_date}')"

    #Final query
    query = f"""
        SELECT
            {select_clause}
        FROM
            {fact_table} {fact_alias}
            {join_clause}
        WHERE 
            {where_clause}
    """.strip()

    return query


## **DF_FACT**

In [0]:
query = generate_fact_query_incremental(fact_table=fact_table, dimensions=dimensions, fact_columns=fact_column, cdc_column=cdc_col, processing_date=last_load_date)

df_fact = spark.sql(query)
df_fact.display()

## Upsert

In [0]:
#Fact Key Column Merge Condition
fact_key_columns_str = " AND ".join([f"src.{col} = tgt.{col}" for col in fact_key_columns])
fact_key_columns_str

In [0]:
from delta.tables import DeltaTable

In [0]:
#Insert data if table is not exist 
#Upsert if the table is exist
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    #df_union.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.{target_schema}.{target_object}")
    dlt_object = DeltaTable.forName(spark, f"{catalog}.{target_schema}.{target_object}")
    dlt_object.alias("tgt").merge(df_fact.alias("src"), fact_key_columns_str)\
        .whenMatchedUpdateAll(condition= f"src.{cdc_col} >= tgt.{cdc_col}")\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    df_fact.write.format("delta").mode("append")\
        .saveAsTable(f"{catalog}.{target_schema}.{target_object}")
        

In [0]:
%sql
SELECT * FROM dataengineerflightproject.gold.fact_bookings